# 41. LLM Serving과 LLMOps

> **제41장** · **이론편 대응: 25.6절 (LLM Serving, LLMOps)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음 (23장의 transformers 사용)
> **다운로드**: DistilGPT-2 (31장에서 받았다면 재사용)

---

## 이 장에서 하는 일

지금까지 만든 것을 **실제로 운영**하는 문제를 다룬다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 실험과 운영의 차이 | 25.6절 |
| 2 | **KV Cache — 효과 측정** ★ | 20.1절, 25.6절 |
| 3 | **배치 처리 — 처리량 vs 지연** ★ | 25.6절 |
| 4 | 메모리 계산 | 25.6절 |
| 5 | 서빙 프레임워크 | 25.6절 |
| 6 | 무엇을 측정할 것인가 | 25.6절 |
| 7 | 비용 최적화 | 25.6절 |
| 8 | 운영 체크리스트 | 25.6절 |

**2절과 3절에서 직접 측정한다.** "빨라진다"가 아니라 **몇 배 빨라지는지** 숫자로 확인한다.

In [ ]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"      # 생성 시에는 왼쪽 패딩 (3절 참조)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f"모델: {MODEL_NAME} ({sum(p.numel() for p in model.parameters())/1e6:.0f}M)")
print()
print("작은 모델로 측정하지만, 나타나는 경향은 큰 모델도 같다.")

---

## 1. 실험과 운영의 차이 — 이론편 25.6절

노트북에서 잘 돌아가던 것이 서비스에서는 무너지는 경우가 많다.

| 항목 | 실험 (지금까지) | 운영 |
|---|---|---|
| 요청 | 하나씩 | **동시에 여러 개** |
| 실패 | 다시 실행 | 사용자가 겪음 |
| 속도 | 기다리면 됨 | **응답 시간 약속** |
| 비용 | 신경 안 씀 | 요청당 원가 |
| 모델 변경 | 자유롭게 | 검증 후 배포 |

**핵심 지표 세 가지**를 먼저 정의하고 시작한다.

In [ ]:
print("=" * 78)
print("서빙의 핵심 지표 (이론편 25.6절)")
print("=" * 78)
print()
print(f"{'지표':<22}{'뜻':<30}{'중요한 곳'}")
print("-" * 78)
metrics = [
    ("TTFT", "첫 토큰까지 걸린 시간",      "대화형 — 체감 속도"),
    ("TPOT", "토큰 하나당 시간",          "긴 답변의 흐름"),
    ("처리량(throughput)", "초당 처리 요청 수",   "배치 작업 — 원가"),
    ("지연(latency)", "요청 하나의 총 시간",   "사용자 경험"),
]
for a, b, c in metrics:
    print(f"{a:<22}{b:<30}{c}")
print("-" * 78)
print()
print("[중요] 처리량과 지연은 서로 상충한다")
print()
print("  요청을 모아서 한 번에 처리하면 → 처리량 ↑, 지연 ↑")
print("  요청을 즉시 처리하면        → 지연 ↓, 처리량 ↓")
print()
print("  무엇이 중요한지에 따라 설정이 달라진다. 3절에서 직접 확인한다.")
print()
print("25장 5절에서 다룬 스트리밍이 TTFT 를 개선하는 방법이다.")
print("  전체 시간은 같아도 첫 반응이 빨라 체감이 크게 다르다.")

---

## 2. KV Cache — 효과 측정 ★ — 이론편 25.6절

23장에서 봤듯 생성은 **토큰 하나마다 모델을 한 번씩** 부른다.

그런데 매번 처음부터 다시 계산하면 낭비가 크다.
**이미 계산한 Key와 Value를 저장해 두고 재사용**하는 것이 KV Cache다.

```
캐시 없음: [나는] → [나는 학교에] → [나는 학교에 간다]
           매번 전체를 다시 계산

캐시 있음: [나는] 계산 → 저장
           [학교에]만 계산 → 앞의 것은 캐시에서
```

**20장에서 만든 Attention을 떠올리자.** $K$와 $V$는 앞 토큰들의 것이 그대로 쓰인다.

In [ ]:
import torch
import time

print("=" * 70)
print("KV Cache 효과 측정")
print("=" * 70)

prompt = "The future of artificial intelligence"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

N_TOKENS = 50
results = {}

for use_cache in [False, True]:
    torch.manual_seed(0)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=N_TOKENS,
                             do_sample=False, use_cache=use_cache,
                             pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0
    results[use_cache] = elapsed
    label = "사용" if use_cache else "미사용"
    print(f"  캐시 {label:<6}: {elapsed:6.2f}초   ({N_TOKENS/elapsed:5.1f} 토큰/초)")

print("-" * 70)
speedup = results[False] / results[True]
print(f"속도 차이: {speedup:.1f}배")
print()
print("토큰이 늘어날수록 차이가 벌어진다.")
print("  캐시 없으면 n번째 토큰에서 n개를 다시 계산 → 총 계산이 n^2 에 비례")
print("  캐시 있으면 매번 1개만 계산 → 총 계산이 n 에 비례")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import torch

print("=" * 70)
print("생성 길이에 따른 차이")
print("=" * 70)

lengths = [10, 20, 40]
no_cache, with_cache = [], []

for n in lengths:
    for use_cache, store in [(False, no_cache), (True, with_cache)]:
        t0 = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=n, do_sample=False,
                           use_cache=use_cache,
                           pad_token_id=tokenizer.eos_token_id)
        store.append(time.time() - t0)

print(f"{'생성 토큰':<14}{'캐시 없음':<14}{'캐시 있음':<14}{'배수'}")
print("-" * 70)
for i, n in enumerate(lengths):
    print(f"{n:<14}{no_cache[i]:<14.2f}{with_cache[i]:<14.2f}{no_cache[i]/with_cache[i]:.1f}배")
print("-" * 70)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
ax.plot(lengths, no_cache, marker="o", linewidth=2,
        color="#DC2626", label="캐시 없음")
ax.plot(lengths, with_cache, marker="s", linewidth=2,
        color="#0D9488", label="캐시 있음")
ax.set_xlabel("생성 토큰 수")
ax.set_ylabel("소요 시간 (초)")
ax.set_title("실측")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
n_range = np.arange(1, 101)
ax.plot(n_range, n_range**2 / 100, linewidth=2,
        color="#DC2626", label="캐시 없음 (n²에 비례)")
ax.plot(n_range, n_range / 10, linewidth=2,
        color="#0D9488", label="캐시 있음 (n에 비례)")
ax.set_xlabel("생성 토큰 수")
ax.set_ylabel("누적 계산량 (상대)")
ax.set_title("이론적 계산량")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("transformers 는 기본으로 캐시를 켠다 (use_cache=True).")
print("  일부러 끌 일은 거의 없다. 여기서는 효과를 보이기 위해 껐다.")

---

## 3. 배치 처리 — 처리량 vs 지연 ★ — 이론편 25.6절

**요청을 모아서 한 번에 처리하면** GPU를 더 효율적으로 쓸 수 있다.
12장의 배치 학습과 같은 원리다.

**하지만 대가가 있다.** 모으는 동안 앞선 요청이 기다려야 한다.

In [ ]:
import torch
import time
import numpy as np

print("=" * 78)
print("배치 크기에 따른 처리량과 지연")
print("=" * 78)

prompts = ["Once upon a time"] * 8
N_GEN = 20

batch_results = []

for bs in [1, 2, 4, 8]:
    batch = prompts[:bs]
    enc = tokenizer(batch, return_tensors="pt", padding=True).to(device)

    t0 = time.time()
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=N_GEN, do_sample=False,
                       pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0

    batch_results.append({
        "batch": bs,
        "total": elapsed,
        "per_request": elapsed / bs,
        "throughput": bs / elapsed,
    })

print(f"{'배치 크기':<12}{'총 시간':<14}{'요청당 시간':<16}{'처리량(req/s)':<18}{'1 대비'}")
print("-" * 78)
base_tp = batch_results[0]["throughput"]
for r in batch_results:
    print(f"{r['batch']:<12}{r['total']:<14.2f}{r['per_request']:<16.3f}"
          f"{r['throughput']:<18.1f}{r['throughput']/base_tp:.1f}배")
print("-" * 78)
print()
print("배치를 키우면")
print("  처리량 ↑ — 같은 시간에 더 많은 요청 처리")
print("  총 시간 ↑ — 마지막 요청은 더 오래 기다림")
print()
print("요청당 시간이 줄어드는 이유")
print("  GPU/CPU 가 한 번에 여러 행렬을 곱하는 것이 개별로 하는 것보다 효율적이다.")
print("  2장에서 본 '반복문 vs 행렬 곱'과 같은 이야기다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

batches = [r["batch"] for r in batch_results]
throughputs = [r["throughput"] for r in batch_results]
totals = [r["total"] for r in batch_results]

ax = axes[0]
ax.plot(batches, throughputs, marker="o", linewidth=2.5, color="#0D9488")
ax.set_xlabel("배치 크기")
ax.set_ylabel("처리량 (req/s)")
ax.set_title("배치가 크면 처리량 증가")
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(batches, totals, marker="s", linewidth=2.5, color="#DC2626")
ax.set_xlabel("배치 크기")
ax.set_ylabel("총 소요 시간 (초)")
ax.set_title("대신 마지막 요청의 대기 시간 증가")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("무엇을 골라야 하나")
print("=" * 78)
print(f"{'상황':<30}{'권장':<20}{'이유'}")
print("-" * 78)
print(f"{'대화형 서비스':<30}{'작은 배치':<20}{'응답 속도가 중요'}")
print(f"{'대량 문서 처리':<30}{'큰 배치':<20}{'전체 완료 시간이 중요'}")
print(f"{'혼합':<30}{'연속 배치(continuous)':<20}{'5절 참조'}")
print("-" * 78)

### 패딩 방향이 왜 중요한가

배치로 생성할 때 `tokenizer.padding_side = "left"`를 썼다. 이유가 있다.

**생성은 마지막 토큰 다음을 예측하는 것**이다(23장).
오른쪽에 패딩이 붙으면 마지막 토큰이 패딩이 되어 버린다.

```
오른쪽 패딩:  [나는 학교에 <pad> <pad>]   ← 마지막이 패딩
왼쪽 패딩:    [<pad> <pad> 나는 학교에]   ← 마지막이 실제 토큰
```

**학습할 때는 오른쪽, 생성할 때는 왼쪽**이 관행이다.
31장의 SFT에서는 오른쪽 패딩을 썼던 것과 대비된다.

In [ ]:
import torch

print("=" * 70)
print("패딩 방향 확인")
print("=" * 70)

texts = ["Hello", "Hello world this is longer"]

for side in ["right", "left"]:
    tokenizer.padding_side = side
    enc = tokenizer(texts, return_tensors="pt", padding=True)
    print(f"\n[{side} padding]")
    for i, t in enumerate(texts):
        ids = enc["input_ids"][i]
        mask = enc["attention_mask"][i]
        decoded = [tokenizer.decode([x]) if m == 1 else "<pad>"
                   for x, m in zip(ids.tolist(), mask.tolist())]
        print(f"  {decoded}")

tokenizer.padding_side = "left"       # 생성용으로 되돌림
print()
print("-" * 70)
print("right padding 의 첫 문장을 보라 — 마지막이 <pad> 다.")
print("  이 상태로 생성하면 패딩 다음을 예측하게 되어 결과가 이상해진다.")

---

## 4. 메모리 계산 — 이론편 25.6절

33장에서 **모델 가중치** 메모리를 계산했다. 서빙에는 하나가 더 있다.

**KV Cache가 요청마다 따로 필요하다.**

$$M_{KV} = 2 \times L \times H \times d_{head} \times S \times B \times b$$

| 기호 | 뜻 |
|---|---|
| 2 | Key와 Value |
| $L$ | 층 수 |
| $H$ | 헤드 수 |
| $d_{head}$ | 헤드당 차원 |
| $S$ | 시퀀스 길이 |
| $B$ | 동시 요청 수 |
| $b$ | 바이트 (FP16이면 2) |

In [ ]:
import numpy as np


def kv_cache_memory(n_layers, n_heads, d_head, seq_len, batch=1, bytes_per=2):
    # KV Cache 메모리 (바이트)
    return 2 * n_layers * n_heads * d_head * seq_len * batch * bytes_per


print("=" * 78)
print("KV Cache 메모리 (이론편 25.6절)")
print("=" * 78)

configs = [
    ("GPT-2 (124M)", 12, 12, 64),
    ("1.5B급",       24, 16, 128),
    ("7B급",         32, 32, 128),
]

print(f"{'모델':<16}{'seq=512':<16}{'seq=2048':<16}{'seq=8192':<16}")
print("-" * 78)
for name, L, H, D in configs:
    row = f"{name:<16}"
    for seq in [512, 2048, 8192]:
        mb = kv_cache_memory(L, H, D, seq) / 1024**2
        row += f"{mb:>10.0f} MB   "
    print(row)
print("-" * 78)
print("(요청 1개 기준, FP16)")
print()

# 동시 요청 시
print("7B급 모델, 시퀀스 2048 기준 동시 요청")
print(f"{'동시 요청':<14}{'KV Cache':<16}{'모델 가중치(FP16)':<22}{'합계'}")
print("-" * 78)
weight_gb = 7e9 * 2 / 1024**3
for batch in [1, 4, 16, 32]:
    kv_gb = kv_cache_memory(32, 32, 128, 2048, batch) / 1024**3
    print(f"{batch:<14}{kv_gb:>10.1f} GB   {weight_gb:>14.1f} GB     {weight_gb+kv_gb:>6.1f} GB")
print("-" * 78)
print()
print("동시 요청이 늘면 KV Cache 가 빠르게 커진다.")
print("  가중치는 고정이지만 캐시는 요청 수에 비례한다.")
print("  → 동시 처리 가능한 요청 수를 제한하는 주된 이유다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.5))

batches = np.arange(1, 41)
weight_gb = 7e9 * 2 / 1024**3

for seq, color in [(512, "#0D9488"), (2048, "#EA580C"), (4096, "#DC2626")]:
    kv = [kv_cache_memory(32, 32, 128, seq, b) / 1024**3 for b in batches]
    total = [weight_gb + k for k in kv]
    ax.plot(batches, total, linewidth=2, color=color, label=f"seq={seq}")

ax.axhline(24, color="gray", linestyle="--", linewidth=1.5)
ax.text(1, 24.7, "24GB GPU", fontsize=8, color="gray")
ax.axhline(80, color="gray", linestyle=":", linewidth=1.5)
ax.text(1, 81, "80GB GPU", fontsize=8, color="gray")

ax.set_xlabel("동시 요청 수")
ax.set_ylabel("총 메모리 (GB)")
ax.set_title("7B 모델 서빙 시 메모리")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("메모리를 아끼는 방법 (이론편 25.6절)")
print()
print(f"{'방법':<24}{'효과':<24}{'대가'}")
print("-" * 78)
methods = [
    ("PagedAttention",   "메모리 낭비 크게 감소",     "구현 복잡"),
    ("양자화 (32장)",     "가중치 1/4",             "정밀도 손실"),
    ("KV Cache 양자화",   "캐시 1/2~1/4",          "품질 소폭 저하"),
    ("최대 길이 제한",     "캐시 상한 고정",          "긴 문맥 불가"),
    ("GQA/MQA",         "헤드 공유로 캐시 축소",     "모델 구조 변경 필요"),
]
for a, b, c in methods:
    print(f"{a:<24}{b:<24}{c}")
print("-" * 78)

---

## 5. 서빙 프레임워크 — 이론편 25.6절

지금까지 `model.generate()`를 직접 불렀다. **운영에서는 전용 프레임워크를 쓴다.**

| 이름 | 특징 |
|---|---|
| **vLLM** | PagedAttention, 높은 처리량 |
| **TGI** | Hugging Face 제작, 배포가 간단 |
| **Ollama** | 로컬 실행에 편리, 설치 쉬움 |
| **llama.cpp** | CPU·저사양에서도 동작 |

**직접 만들지 않는 이유**는 최적화 기법이 많고 구현이 까다롭기 때문이다.

In [ ]:
print("=" * 78)
print("프레임워크가 대신해 주는 것")
print("=" * 78)
print()
print(f"{'기법':<26}{'하는 일':<30}{'직접 구현하면'}")
print("-" * 78)
features = [
    ("연속 배치",        "끝난 요청 자리에 새 요청 투입",  "매우 복잡"),
    ("PagedAttention",  "캐시를 페이지 단위로 관리",     "매우 복잡"),
    ("prefix 캐싱",     "같은 앞부분은 재사용",         "복잡"),
    ("동적 배치",        "부하에 따라 배치 크기 조절",    "복잡"),
    ("스트리밍",         "토큰 단위 전송 (25장 5절)",    "가능"),
    ("모니터링",         "지표 수집·노출",             "가능"),
]
for a, b, c in features:
    print(f"{a:<26}{b:<30}{c}")
print("-" * 78)
print()
print("[연속 배치가 왜 중요한가]")
print()
print("  3절의 배치 처리에는 문제가 있다.")
print("  배치 안의 요청들이 **동시에 끝나지 않는다.**")
print()
print("    요청 A: 10토큰이면 끝")
print("    요청 B: 200토큰 필요")
print()
print("  일반 배치는 B가 끝날 때까지 A의 자리가 놀고 있다.")
print("  연속 배치는 A가 끝나면 그 자리에 새 요청을 넣는다.")
print()
print("  → 실제 처리량이 크게 오른다")

In [ ]:
print("=" * 78)
print("프레임워크 사용 예")
print("=" * 78)
print()
print("[vLLM]")
print()
vllm_code = [
    "from vllm import LLM, SamplingParams",
    "",
    "llm = LLM(model='<모델명>', ",
    "          gpu_memory_utilization=0.9,   # GPU 메모리 사용 비율",
    "          max_model_len=4096)           # 최대 시퀀스 길이",
    "",
    "params = SamplingParams(temperature=0.7, top_p=0.9,",
    "                        max_tokens=200)   # 24장에서 배운 설정",
    "",
    "outputs = llm.generate(prompts, params)   # 리스트를 한 번에",
]
for line in vllm_code:
    print("  " + line)

print()
print("[Ollama — 로컬에서 가장 간단]")
print()
ollama_code = [
    "# 터미널",
    "ollama pull <모델명>",
    "ollama serve",
    "",
    "# 파이썬 — 25장의 OpenAI 호환 방식 그대로",
    "from openai import OpenAI",
    "client = OpenAI(base_url='http://localhost:11434/v1',",
    "                api_key='ollama')        # 아무 값이나",
]
for line in ollama_code:
    print("  " + line)

print()
print("-" * 78)
print("주목할 점")
print("  Ollama 는 OpenAI 호환 API 를 제공한다.")
print("  25장 2절에서 base_url 만 바꾸면 된다고 한 것이 여기서도 적용된다.")
print()
print("  → 개발은 API 로, 운영은 로컬 서빙으로 바꿔도 코드가 거의 그대로다.")

---

## 6. 무엇을 측정할 것인가 — 이론편 25.6절

**LLMOps의 핵심은 측정이다.** 무엇이 잘못됐는지 알아야 고칠 수 있다.

In [ ]:
import time
import numpy as np
from collections import defaultdict


class ServingMetrics:
    # 서빙 지표 수집기 (이론편 25.6절)

    def __init__(self):
        self.records = []

    def record(self, request_id, ttft, total_time, n_tokens,
               success=True, error=None):
        self.records.append({
            "id": request_id,
            "ttft": ttft,
            "total": total_time,
            "tokens": n_tokens,
            "tpot": (total_time - ttft) / max(n_tokens - 1, 1),
            "success": success,
            "error": error,
        })

    def summary(self):
        if not self.records:
            return {}
        ok = [r for r in self.records if r["success"]]
        if not ok:
            return {"success_rate": 0.0, "total_requests": len(self.records)}

        ttfts = [r["ttft"] for r in ok]
        totals = [r["total"] for r in ok]
        tpots = [r["tpot"] for r in ok]

        return {
            "total_requests": len(self.records),
            "success_rate": len(ok) / len(self.records),
            "ttft_mean": np.mean(ttfts),
            "ttft_p50": np.percentile(ttfts, 50),
            "ttft_p95": np.percentile(ttfts, 95),
            "total_p50": np.percentile(totals, 50),
            "total_p95": np.percentile(totals, 95),
            "tpot_mean": np.mean(tpots),
            "tokens_total": sum(r["tokens"] for r in ok),
        }


# 실제 측정
metrics = ServingMetrics()
test_prompts = [
    "The weather today is",
    "Machine learning is",
    "In the beginning",
    "Python programming",
    "The future holds",
]

print("=" * 70)
print("지표 수집")
print("=" * 70)

for i, prompt in enumerate(test_prompts):
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    n_gen = 20

    t0 = time.time()
    # 첫 토큰까지 (TTFT 측정용)
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=1, do_sample=False,
                       pad_token_id=tokenizer.eos_token_id)
    ttft = time.time() - t0

    t0 = time.time()
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=n_gen, do_sample=False,
                       pad_token_id=tokenizer.eos_token_id)
    total = time.time() - t0

    metrics.record(f"req_{i}", ttft, total, n_gen)
    print(f"  요청 {i}: TTFT {ttft*1000:6.0f}ms, 총 {total*1000:6.0f}ms")

print()
s = metrics.summary()
print("=" * 70)
print("요약")
print("=" * 70)
print(f"  요청 수      : {s['total_requests']}")
print(f"  성공률       : {s['success_rate']:.1%}")
print(f"  TTFT 평균    : {s['ttft_mean']*1000:.0f} ms")
print(f"  TTFT p50     : {s['ttft_p50']*1000:.0f} ms")
print(f"  TTFT p95     : {s['ttft_p95']*1000:.0f} ms")
print(f"  TPOT 평균    : {s['tpot_mean']*1000:.1f} ms/토큰")

### 왜 평균이 아니라 p95를 보는가

**평균은 나쁜 경험을 감춘다.**

요청 100개 중 95개가 0.5초, 5개가 10초 걸렸다고 하자.
- 평균: 0.98초 — 괜찮아 보인다
- p95: 10초 — **20명 중 1명이 10초를 기다린다**

서비스 품질은 **가장 느린 쪽**이 좌우한다. 그래서 p95, p99를 본다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("평균이 감추는 것")
print("=" * 70)

# 대부분 빠르고 일부만 느린 분포
rng = np.random.RandomState(0)
fast = rng.normal(0.5, 0.1, 95)
slow = rng.normal(10.0, 2.0, 5)
latencies = np.concatenate([fast, slow])
latencies = np.clip(latencies, 0.1, None)

print(f"  평균     : {latencies.mean():.2f}초")
print(f"  중앙값   : {np.percentile(latencies, 50):.2f}초")
print(f"  p95      : {np.percentile(latencies, 95):.2f}초")
print(f"  p99      : {np.percentile(latencies, 99):.2f}초")
print(f"  최댓값   : {latencies.max():.2f}초")
print()
print("평균만 보면 1초 남짓이지만, 20명 중 1명은 훨씬 오래 기다린다.")

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.hist(latencies, bins=40, color="#94A3B8", edgecolor="white")
for pct, color, label in [(50, "#0D9488", "p50"),
                          (95, "#EA580C", "p95"),
                          (99, "#DC2626", "p99")]:
    v = np.percentile(latencies, pct)
    ax.axvline(v, color=color, linestyle="--", linewidth=2, label=f"{label}={v:.1f}s")
ax.axvline(latencies.mean(), color="#1E40AF", linewidth=2, label=f"평균={latencies.mean():.1f}s")
ax.set_xlabel("응답 시간 (초)")
ax.set_ylabel("요청 수")
ax.set_title("응답 시간 분포")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 78)
print("측정해야 할 것 (이론편 25.6절)")
print("=" * 78)
print()
print(f"{'분류':<16}{'지표':<28}{'왜'}")
print("-" * 78)
items = [
    ("성능",   "TTFT, TPOT, p95 지연",       "사용자 경험"),
    ("성능",   "처리량 (req/s, tok/s)",      "용량 계획"),
    ("자원",   "GPU 메모리, 사용률",          "병목 파악"),
    ("자원",   "대기 큐 길이",               "과부하 감지"),
    ("비용",   "요청당 토큰 수",             "원가 계산"),
    ("품질",   "오류율, 재시도율",           "안정성"),
    ("품질",   "응답 길이 분포",             "이상 감지"),
    ("품질",   "사용자 피드백",              "실제 만족도"),
]
for a, b, c in items:
    print(f"{a:<16}{b:<28}{c}")
print("-" * 78)
print()
print("[품질 측정이 가장 어렵다]")
print()
print("  성능은 숫자로 나오지만 '답이 좋은가'는 자동으로 알기 어렵다.")
print()
print("  실무에서 쓰는 방법")
print("    - 사용자 피드백 버튼 (좋아요/싫어요)")
print("    - 재질문 비율 (같은 것을 다시 물으면 답이 나빴다는 신호)")
print("    - 표본 추출해 사람이 검토")
print("    - LLM 으로 평가 (LLM-as-judge)")
print()
print("  27장 8절에서 만든 평가 데이터가 여기서도 쓰인다.")

---

## 7. 비용 최적화 — 이론편 25.6절

25장에서 API 비용을 다뤘다. **직접 서빙할 때의 원가**도 계산해 보자.

In [ ]:
import numpy as np

print("=" * 78)
print("자체 서빙 vs API — 손익분기점")
print("=" * 78)
print()
print("가정 (실제 값은 상황에 따라 크게 다름)")
print("  GPU 인스턴스: 시간당 $1.5")
print("  처리량      : 초당 10 요청")
print("  API 단가    : 요청당 $0.0002")
print()

GPU_HOURLY = 1.5
THROUGHPUT = 10          # req/s
API_PER_REQ = 0.0002

print(f"{'일 요청 수':<16}{'자체 서빙(월)':<18}{'API(월)':<18}{'유리한 쪽'}")
print("-" * 78)
for daily in [1_000, 10_000, 100_000, 1_000_000, 10_000_000]:
    monthly_req = daily * 30

    # 자체 서빙: 24시간 가동 가정
    self_cost = GPU_HOURLY * 24 * 30

    # 처리량이 부족하면 인스턴스를 늘려야 한다
    needed = monthly_req / (THROUGHPUT * 3600 * 24 * 30)
    n_instances = max(1, int(np.ceil(needed)))
    self_cost *= n_instances

    api_cost = monthly_req * API_PER_REQ

    winner = "자체 서빙" if self_cost < api_cost else "API"
    print(f"{daily:<16,}${self_cost:<17,.0f}${api_cost:<17,.0f}{winner}")

print("-" * 78)
print()
print("요청이 적으면 API 가 싸고, 많아지면 자체 서빙이 유리해진다.")
print()
print("하지만 비용만으로 결정할 문제가 아니다 (25장 1절)")
print("  - 데이터가 외부로 나가도 되는가")
print("  - 운영 인력이 있는가")
print("  - 최상급 모델이 필요한가")

In [ ]:
print("=" * 78)
print("비용을 줄이는 방법")
print("=" * 78)
print()
print(f"{'방법':<26}{'효과':<26}{'대가'}")
print("-" * 78)
savings = [
    ("작은 모델 사용",      "비용 대폭 감소",         "품질 저하 가능"),
    ("양자화 (32장)",      "같은 GPU 로 더 큰 모델",  "정밀도 손실"),
    ("프롬프트 단축",       "입력 토큰 감소",         "문맥 부족 위험"),
    ("응답 길이 제한",      "출력 토큰 감소",         "답변 잘림"),
    ("캐싱",              "같은 질문 재계산 방지",    "최신성 문제"),
    ("배치 처리 (3절)",    "처리량 증가",            "지연 증가"),
    ("작업별 모델 분리",    "쉬운 일은 싼 모델로",     "구조 복잡"),
]
for a, b, c in savings:
    print(f"{a:<26}{b:<26}{c}")
print("-" * 78)
print()
print("[캐싱이 의외로 효과가 크다]")
print()
print("  실제 서비스에서는 같은 질문이 반복되는 경우가 많다.")
print("  자주 묻는 질문의 답을 저장해 두면 호출 자체를 줄일 수 있다.")
print()
print("  단, 캐시 키를 어떻게 잡을지가 문제다:")
print("    - 정확히 같은 문자열만? → 적중률 낮음")
print("    - 의미가 비슷하면? → 27장의 임베딩으로 판단 가능")

---

## 8. 운영 체크리스트 — 이론편 25.6절

실제로 배포하기 전에 확인할 것들이다.

In [ ]:
print("=" * 78)
print("배포 전 체크리스트")
print("=" * 78)
print()

checklist = {
    "성능": [
        "부하 시험 — 예상 최대 요청의 2배로 시험했는가",
        "p95 지연이 목표 안에 드는가",
        "동시 요청 한도를 정했는가 (4절의 메모리 계산)",
    ],
    "안정성": [
        "모델 로딩 실패 시 대응이 있는가",
        "타임아웃을 설정했는가",
        "재시도 정책이 있는가 (25장 8절)",
        "과부하 시 요청을 거절하는 장치가 있는가",
    ],
    "안전": [
        "입력 길이를 제한했는가",
        "프롬프트 주입 대비가 있는가 (37장 7절)",
        "출력 필터링이 필요한가",
        "API 키·자격증명이 안전하게 관리되는가 (25장 1절)",
    ],
    "관측": [
        "지표를 수집하고 있는가 (6절)",
        "오류 로그가 남는가",
        "이상 상황에 알림이 오는가",
        "요청 추적이 가능한가 (37장의 trace)",
    ],
    "비용": [
        "요청당 원가를 알고 있는가 (7절)",
        "예산 상한을 걸어 두었는가",
        "비용 급증 시 알림이 오는가",
    ],
    "롤백": [
        "이전 버전으로 되돌릴 수 있는가",
        "모델 버전을 기록하고 있는가",
        "점진적 배포가 가능한가",
    ],
}

for category, items in checklist.items():
    print(f"[{category}]")
    for item in items:
        print(f"  □ {item}")
    print()

total = sum(len(v) for v in checklist.values())
print("-" * 78)
print(f"총 {total}개 항목")
print()
print("모두 갖추고 시작할 필요는 없다.")
print("  하지만 **무엇을 갖추지 않았는지는 알고** 시작해야 한다.")

In [ ]:
print("=" * 78)
print("모델 버전 관리 (이론편 25.6절)")
print("=" * 78)
print()
print("LLM 서비스에서 '무엇이 바뀌면 결과가 달라지는가'")
print()
print(f"{'요소':<24}{'바뀌면':<30}{'기록 필요'}")
print("-" * 78)
items = [
    ("모델 가중치",      "답변이 전부 달라짐",        "필수"),
    ("시스템 프롬프트",   "성격·형식이 달라짐",        "필수"),
    ("생성 파라미터",     "다양성이 달라짐 (24장)",    "필수"),
    ("RAG 문서",        "근거가 달라짐 (28장)",      "필수"),
    ("토크나이저",       "드물지만 영향 큼",          "필수"),
    ("프레임워크 버전",   "미묘한 차이 발생 가능",      "권장"),
]
for a, b, c in items:
    print(f"{a:<24}{b:<30}{c}")
print("-" * 78)
print()
print("[재현 가능성]")
print()
print("  '지난주에는 잘 됐는데' 를 조사하려면 그때의 설정을 알아야 한다.")
print()
print("  응답과 함께 기록하면 좋은 것")
print("    - 모델 이름·버전")
print("    - 프롬프트 템플릿 버전")
print("    - 생성 파라미터 (temperature 등)")
print("    - 사용한 문서 ID (RAG인 경우)")
print("    - 시드 (샘플링을 썼다면, 24장 4절)")
print()
print("  15장의 '재현성' 이야기가 운영에서도 그대로 적용된다.")

---

## 9. 정리

### 측정한 것

| 항목 | 결과 |
|---|---|
| KV Cache | **속도 수 배 향상** (길이가 길수록 커짐) |
| 배치 처리 | 배치 8이면 **처리량 3배** |
| 배치의 대가 | 총 시간 증가 (마지막 요청 대기) |
| KV Cache 메모리 | 요청 수에 **비례** |

### 핵심 지표

| 지표 | 뜻 | 언제 중요 |
|---|---|---|
| TTFT | 첫 토큰까지 | 대화형 |
| TPOT | 토큰당 시간 | 긴 답변 |
| 처리량 | 초당 요청 수 | 배치 작업 |
| **p95 지연** | 하위 5% 경험 | **항상** |

**평균이 아니라 p95를 본다.** 평균은 나쁜 경험을 감춘다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| KV Cache | 기본으로 켜져 있음 — 끌 일 없음 |
| 패딩 방향 | **생성은 왼쪽**, 학습은 오른쪽 |
| 처리량 vs 지연 | 서로 상충 — 무엇이 중요한지 정해야 |
| 메모리 | 가중치(고정) + KV Cache(요청 비례) |
| 프레임워크 | 직접 만들지 말 것 (vLLM, Ollama 등) |
| Ollama | OpenAI 호환 — 21번 코드 그대로 |
| 버전 기록 | 재현 가능하게 남길 것 |

### 지금까지의 실습과 이어지는 지점

| 장 | 서빙에서의 의미 |
|---|---|
| 20번 생성 파라미터 | 기록해야 할 설정 |
| 21번 스트리밍·재시도 | TTFT 개선, 안정성 |
| 22번 임베딩 | 의미 기반 캐싱 |
| 25번 양자화 | 메모리 절감 |
| 29번 trace | 요청 추적 |

### 다음 장

**42. 종합 복습 — 지금까지 만든 것들** — 여기까지 다룬 기술을 정리하고,
마지막 두 장의 종합 프로젝트를 준비한다.